In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
from torch import Tensor
import torch.nn.functional as F
from torch.nn import init, Parameter
import pdb
import math
import numpy as np
import torch.nn.functional as F
from torch.autograd import Variable
import os 
import random
import pandas as pd
from copy import deepcopy
from typing import Dict
import transformers
from torch import Tensor
from torch.nn import init, Parameter
import torch.nn.functional as F
import pdb
import math
from collections import defaultdict
import numpy as np
import os 
import json
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, f1_score, average_precision_score, precision_score
from torch.utils.data import DataLoader,Dataset,TensorDataset
import torch.optim as optim

In [2]:
hours_data = 24
data_path = "physionet.org/files/challenge-2012/1.0.0/phase1/all_subjects_data_48_hours/"
subject= np.load(os.path.join(data_path,"subjects_data.npz"),allow_pickle=True)
S = np.squeeze(subject["statics_data"], axis=1)
targets = subject["targets_data"]
timeseries = np.array(subject["timeseries_imp_data"], dtype=float)[:, :hours_data, :]
timeseries_org = np.array(subject["timeseries_data"], dtype=float)[:, :hours_data, :]
timedelta = np.array(subject["delta_time_data"], dtype=float)[:, :hours_data, :]
icu_mor = targets

In [3]:
unique, counts = np.unique(icu_mor, return_counts=True)

dict(zip(unique, counts))

{np.int64(0): np.int64(10281), np.int64(1): np.int64(1707)}

In [4]:
repeats =timeseries.shape[1]
temporal_statics = np.tile(S, repeats).reshape(S.shape[0], repeats,S.shape[-1])
timedelta_statics= np.zeros((S.shape[0], repeats,S.shape[-1]))
temporal_statics.shape, timedelta_statics.shape

((11988, 24, 2), (11988, 24, 2))

In [5]:
temporal_data = np.concatenate((timeseries_org, temporal_statics), axis=-1)
temporal_timedelta = np.concatenate((timedelta, np.zeros_like(temporal_statics)), axis=-1)
temporal_timedelta[np.isnan(temporal_timedelta)] = 999
temporal_timedelta[np.isinf(temporal_timedelta)] = 999
temporal_data_features = np.concatenate((timeseries_org, temporal_statics), axis=-1)

In [6]:
temporal_data_features

array([[[34.  ,   nan, 12.  , ...,  7.32, 17.  ,  7.  ],
        [34.  ,   nan, 11.  , ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        ...,
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ]],

       [[42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,  3.1 , 41.  , ...,   nan,  4.  , 10.  ],
        ...,
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ]],

       [[79.  ,   nan,   nan, ...,  7.43, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,  7.32, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,  7.39, 18.  ,  8.  ],
        ...,
        [79.  ,   nan,   nan, ...,   nan, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,   nan, 18.

In [7]:
timeseries.shape,timeseries_org.shape, temporal_data_features.shape

((11988, 24, 26), (11988, 24, 26), (11988, 24, 28))

In [8]:
temporal_data

array([[[34.  ,   nan, 12.  , ...,  7.32, 17.  ,  7.  ],
        [34.  ,   nan, 11.  , ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        ...,
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ],
        [34.  ,   nan,   nan, ...,   nan, 17.  ,  7.  ]],

       [[42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,  3.1 , 41.  , ...,   nan,  4.  , 10.  ],
        ...,
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ],
        [42.  ,   nan,   nan, ...,   nan,  4.  , 10.  ]],

       [[79.  ,   nan,   nan, ...,  7.43, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,  7.32, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,  7.39, 18.  ,  8.  ],
        ...,
        [79.  ,   nan,   nan, ...,   nan, 18.  ,  8.  ],
        [79.  ,   nan,   nan, ...,   nan, 18.

In [13]:
freq_list , timeseries_last_obs_data= [], []
nb_pats, seq, n_features = temporal_data.shape
for i in range(nb_pats):
    data_patient=  np.expand_dims(temporal_data[i,:,:], axis=0)
    nan_counts = np.sum(np.isnan(data_patient), axis=(0, 1))
    freq_list.append(np.repeat(np.expand_dims(nan_counts, axis=0), repeats, axis=0))
    # Last observation record
    Index_Last=(~np.isnan(temporal_data[i,:,:])).cumsum(0).argmax(0)
    Last_Indices = np.reshape(Index_Last,(1,n_features))
    Last_Values = np.take_along_axis(temporal_data[i,:,:], Last_Indices, axis = 0)
    timeseries_last_obs_data.append(np.repeat(Last_Values, repeats, axis=0))
freqs = np.stack(freq_list)
last_obs_data=np.stack(timeseries_last_obs_data)

In [11]:
freqs=1

In [14]:
freqs

array([[[ 0, 24, 21, ..., 17,  0,  0],
        [ 0, 24, 21, ..., 17,  0,  0],
        [ 0, 24, 21, ..., 17,  0,  0],
        ...,
        [ 0, 24, 21, ..., 17,  0,  0],
        [ 0, 24, 21, ..., 17,  0,  0],
        [ 0, 24, 21, ..., 17,  0,  0]],

       [[ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        ...,
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0]],

       [[ 0, 24, 22, ..., 17,  0,  0],
        [ 0, 24, 22, ..., 17,  0,  0],
        [ 0, 24, 22, ..., 17,  0,  0],
        ...,
        [ 0, 24, 22, ..., 17,  0,  0],
        [ 0, 24, 22, ..., 17,  0,  0],
        [ 0, 24, 22, ..., 17,  0,  0]],

       ...,

       [[ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        ...,
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0,  0],
        [ 0, 23, 23, ..., 24,  0

In [19]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset,TensorDataset
from sklearn.model_selection import KFold, train_test_split

def dataloaders_double_cv_cohort(train_x, train_t,  train_last,  train_freq, train_y, 
                          test_x, test_t,  test_last,  test_freq,  test_y, BATCH_SIZE=64):
        
    def collate_fn(batch):
        return tuple(zip(*batch))

    train_dataset = TensorDataset(torch.tensor(train_x,dtype=torch.float32),
                                  torch.tensor(train_t,dtype=torch.float32),
                                  torch.tensor(train_last,dtype=torch.float32),
                                  torch.tensor(train_freq,dtype=torch.float32),
                                  torch.tensor(train_y,dtype=torch.float32))  
    
    
    valid_dataset = TensorDataset(torch.tensor(test_x,dtype=torch.float32),
                                  torch.tensor(test_t,dtype=torch.float32),
                                  torch.tensor(test_last,dtype=torch.float32),
                                  torch.tensor(test_freq,dtype=torch.float32),
                                  torch.tensor(test_y,dtype=torch.float32))  
    
    
    
    train_data_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
    )

    valid_data_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
    )
    
    return train_data_loader, valid_data_loader

def cv_fold_splits_cohorts(data, time_data, last_data,
                           features_freqs, target, n_fold=10):
    from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

    test_data_loader, training_data, validation_data = [], [], []

    kfold = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=n_fold)
    strat_target = target.squeeze() if target.ndim > 1 else target

    # ── StratifiedShuffleSplit for the inner validation split ─────────
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=n_fold)

    for index, (train_index, test_index) in enumerate(
            kfold.split(data, strat_target)):
        print(f'<------- OUTER FOLD {index + 1} ------->')

        # ── Raw outer split (un-normalized) ──────────────────────────
        x_train,      x_test      = data[train_index],          data[test_index]
        x_train_last, x_test_last = last_data[train_index],     last_data[test_index]
        x_train_freq, x_test_freq = features_freqs[train_index],features_freqs[test_index]
        x_train_t,    x_test_t    = time_data[train_index],     time_data[test_index]
        y_train,      y_test      = target[train_index],        target[test_index]

        # ── Inner split via StratifiedShuffleSplit (mirrors cv_fold_splits) ──
        sss.get_n_splits(x_train, y_train)
        tr_idx, val_idx = next(sss.split(x_train, y_train))   # ← changed

        x_tr_in,    x_val_in    = x_train[tr_idx],       x_train[val_idx]
        t_tr_in,    t_val_in    = x_train_t[tr_idx],     x_train_t[val_idx]
        last_tr_in, last_val_in = x_train_last[tr_idx],  x_train_last[val_idx]
        freq_tr_in, freq_val_in = x_train_freq[tr_idx],  x_train_freq[val_idx]
        y_tr_in,    y_val_in    = y_train[tr_idx],        y_train[val_idx]

        # ── Fit normalization ONLY on the inner training split ────────
        def normalize(x, mn, mx):
            scale = np.where(mx - mn == 0, 1.0, mx - mn)
            return (x - mn) / scale

        ts_min = np.nanmin(x_tr_in,    axis=0);  ts_max = np.nanmax(x_tr_in,    axis=0)
        la_min = np.nanmin(last_tr_in, axis=0);  la_max = np.nanmax(last_tr_in, axis=0)
        y_min  = y_tr_in.min();                  y_max  = y_tr_in.max()

        x_tr_in     = normalize(x_tr_in,     ts_min, ts_max)
        x_val_in    = normalize(x_val_in,    ts_min, ts_max)
        x_te_norm   = normalize(x_test,      ts_min, ts_max)

        last_tr_in  = normalize(last_tr_in,  la_min, la_max)
        last_val_in = normalize(last_val_in, la_min, la_max)
        x_test_last = normalize(x_test_last, la_min, la_max)

        y_tr_in  #= normalize(y_tr_in,  y_min, y_max)
        y_val_in #= normalize(y_val_in, y_min, y_max)
        y_test   #= normalize(y_test,   y_min, y_max)

        # ── Build loaders ─────────────────────────────────────────────
        train_loader, _ = dataloaders_double_cv_cohort(
            x_tr_in, t_tr_in, last_tr_in, freq_tr_in, y_tr_in,
            x_tr_in, t_tr_in, last_tr_in, freq_tr_in, y_tr_in
        )

        _, val_loader = dataloaders_double_cv_cohort(
            x_tr_in,  t_tr_in,  last_tr_in,  freq_tr_in,  y_tr_in,
            x_val_in, t_val_in, last_val_in, freq_val_in, y_val_in
        )

        _, test_loader = dataloaders_double_cv_cohort(
            x_tr_in,   t_tr_in,   last_tr_in,  freq_tr_in,  y_tr_in,
            x_te_norm, x_test_t, x_test_last, x_test_freq, y_test
        )

        training_data.append(train_loader)
        validation_data.append(val_loader)
        test_data_loader.append(test_loader)

    return test_data_loader, training_data, validation_data, x_tr_in.shape, ts_min, ts_max, y_min, y_max
    

In [20]:
test_Dataloader, train_Dataloader, valid_Dataloader, data_shape, ts_min, ts_max, y_min , y_max = cv_fold_splits_cohorts(temporal_data_features,
                                                                                                                         temporal_timedelta,
                                                                                                                         last_obs_data, 
                                                                                                                         new_arr,icu_mor,
                                                                                                                         n_fold=10)

<------- OUTER FOLD 1 ------->
<------- OUTER FOLD 2 ------->
<------- OUTER FOLD 3 ------->
<------- OUTER FOLD 4 ------->
<------- OUTER FOLD 5 ------->
<------- OUTER FOLD 6 ------->
<------- OUTER FOLD 7 ------->
<------- OUTER FOLD 8 ------->
<------- OUTER FOLD 9 ------->
<------- OUTER FOLD 10 ------->


In [ ]:
ts_min, ts_max, y_min , y_max

In [21]:
dn=f"Documents/MIMICIII"
if not os.path.exists(dn):
    os.makedirs(dn)

In [ ]:
seq_length = temporal_data_features.shape[1]
input_dim = temporal_data_features.shape[-1]
hidden_dim, output_dim  = 64, icu_mor.shape[-1]
seq_length, input_dim, hidden_dim, output_dim, data_shape
seq_length, input_dim, hidden_dim, output_dim, data_shape
taskname=f"PHYSIONET2012_ICU_FIRST_ALL_SUBJECTS_{seq_length}_HOURS_DATA_10_FOLDS_DyFAIP_FREQS_SET_ZERO".upper()
task=f"{os.path.join(dn, f'{taskname}')}"
if not os.path.exists(task):
    os.makedirs(task)
task

In [23]:
icu_mor.max()

np.int64(1)

In [24]:
train_val_loader = random.sample(list(zip(train_Dataloader, valid_Dataloader)), len(train_Dataloader))
np.savez(os.path.join(task, f"train_test_data.npz"), 
            folds_data_test= test_Dataloader,
            folds_data_train_valid= train_val_loader,)
        
np.savez(os.path.join(task, f"data_max_min.npz"), 
         data_max=ts_max, data_min=ts_min,
         data_targets_max=icu_mor.max(), data_targets_min=icu_mor.min(),
         shape_data=data_shape,seq_length=seq_length,
         input_dim=input_dim, output_dim=output_dim)
input_dim

28

In [ ]:
ts_min, ts_max, y_min , y_max

In [ ]:
unique, counts = np.unique(icu_mor, return_counts=True)

dict(zip(unique, counts))